# Group Project Task Allocation Guide
# Who Does What — Step by Step

---

This notebook breaks down the **exact tasks** each team member is responsible for.

| Member | Role | Main Sections |
|--------|------|---------------|
| **Member A** | Data & Infrastructure | Data collection, cleaning, Alpaca setup |
| **Member B** | Strategy Development | Trading strategies, backtesting |
| **Member C** | Analysis & Reporting | Risk analysis, visualizations, reports |

### How to Use This Notebook
Each section below tells one member **exactly what to do**, with code templates and explanations.

### Timeline
| Week | Member A | Member B | Member C |
|------|----------|----------|----------|
| Week 1 (Mar 15-21) | Collect & clean all data | Design strategies on paper | Set up report template |
| Week 2 (Mar 22-28) | Set up Alpaca paper trading | Implement & backtest strategies | Build visualizations & risk metrics |
| Week 3 (Mar 29-31) | Help debug, final notebook cleanup | Final tuning, generate results | Write & finalize report |
| Week 4 (Apr 1-7) | Monitor paper trading | — | Write follow-up report |

---
---
# MEMBER A: Data & Infrastructure
---
---

## Your Role in Plain English

You are the **foundation builder**. Without clean data, nothing else works.

Think of it like cooking:
- Member B is the chef (builds the recipe/strategy)
- Member C is the food critic (analyzes and writes the review)
- **You buy the ingredients, wash them, chop them, and set up the kitchen**

## Your Tasks (in order):

1. Install all required packages
2. Download stock price data
3. Clean the data (handle missing values, format dates)
4. Calculate technical indicators (RSI, MACD, etc.)
5. Prepare data in the format needed by Member B
6. Set up the Alpaca paper trading connection
7. Organize the final Jupyter notebook

## Task A1: Install Packages

**What are packages?**
Packages are pre-built tools that other programmers have created. Instead of writing everything from scratch, we install these tools and use them.

It's like downloading an app on your phone — someone else built it, you just use it.

**What to do:** Run the cell below. If you see errors, read the error message and Google it.

In [ ]:
# MEMBER A: Run this cell first to install everything
# The '!' tells Jupyter to run this as a terminal command, not Python
# '-q' means quiet (less output)

!pip install -q yfinance       # Downloads stock data from Yahoo Finance (free)
!pip install -q pandas         # Data tables (like Excel in Python)
!pip install -q numpy          # Math operations
!pip install -q matplotlib     # Charts and graphs
!pip install -q seaborn        # Prettier charts
!pip install -q scipy          # Advanced math for portfolio optimization
!pip install -q ta             # Technical indicators (RSI, MACD, etc.)
!pip install -q openai         # For LLM strategy (Path B)
!pip install -q alpaca-trade-api  # Paper trading
!pip install -q finrl          # Reinforcement learning (Path A)

print('All packages installed!')

## Task A2: Download Stock Data

**What is stock data?**
Every trading day, stocks have these values:
- **Open**: Price when market opens (9:30 AM)
- **High**: Highest price during the day
- **Low**: Lowest price during the day  
- **Close**: Price when market closes (4:00 PM) — **this is the most important one**
- **Volume**: Number of shares traded (more volume = more activity)
- **Adjusted Close**: Close price corrected for dividends and stock splits

**What is a stock split?**
If Apple's stock is $200 and they do a 2-for-1 split, the price drops to $100 but you own 2x shares. Your total value doesn't change. Adjusted Close accounts for this.

**What stocks should we pick?**
We need stocks from **different sectors** (industries). This is called **diversification** — if tech crashes, healthcare might be fine.

**What to do:**
1. Choose 10-20 stocks from different sectors
2. Download 5 years of daily data
3. Also download SPY (S&P 500 ETF) as our benchmark

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# =============================================
# MEMBER A: CONFIGURE THESE SETTINGS
# =============================================

# Step 1: Choose your stocks
# Pick 10-20 stocks from DIFFERENT sectors
# A "ticker" is a stock's nickname (e.g., AAPL = Apple)
# You can find tickers by searching on Yahoo Finance

TICKERS = [
    # Technology
    'AAPL',   # Apple
    'MSFT',   # Microsoft
    'GOOGL',  # Google
    'NVDA',   # NVIDIA (chips/AI)
    
    # Finance
    'JPM',    # JPMorgan Chase (bank)
    'V',      # Visa (payments)
    
    # Healthcare
    'JNJ',    # Johnson & Johnson
    'UNH',    # UnitedHealth
    'PFE',    # Pfizer
    
    # Consumer
    'AMZN',   # Amazon
    'PG',     # Procter & Gamble (household goods)
    'KO',     # Coca-Cola
    'HD',     # Home Depot
    
    # Energy
    'XOM',    # ExxonMobil (oil)
    
    # Entertainment
    'DIS',    # Disney
]

BENCHMARK = 'SPY'  # S&P 500 ETF — the benchmark we need to beat

# Step 2: Choose your time period
START_DATE = '2020-01-01'  # 5 years of data
END_DATE   = '2025-12-31'
SPLIT_DATE = '2024-07-01'  # Everything before = training, after = testing

# Step 3: Starting capital
INITIAL_CAPITAL = 1_000_000  # $1 million (fake money)

print(f'Stocks selected: {len(TICKERS)}')
print(f'Sectors covered: Technology, Finance, Healthcare, Consumer, Energy, Entertainment')
print(f'Time period: {START_DATE} to {END_DATE}')
print(f'Train/Test split: {SPLIT_DATE}')

In [ ]:
# MEMBER A: Download the data
# This connects to Yahoo Finance and downloads price history
# It's completely free — no API key needed

print('Downloading stock data from Yahoo Finance...')
print('(This may take 30-60 seconds)\n')

all_tickers = TICKERS + [BENCHMARK]
raw_data = yf.download(all_tickers, start=START_DATE, end=END_DATE, auto_adjust=False)

# Quick check: did it work?
print(f'\nDownload complete!')
print(f'Shape: {raw_data.shape}')  # (rows, columns)
print(f'Date range: {raw_data.index[0].date()} to {raw_data.index[-1].date()}')
print(f'Trading days: {len(raw_data)}')

# Show the first few rows
print('\nSample data (first 3 rows):')
raw_data.head(3)

## Task A3: Clean the Data

**Why do we need to clean data?**
Real-world data is messy:
- Some stocks might not have data for certain days (holidays, listing date)
- There might be missing values (NaN = "Not a Number")
- Dates might be in different formats

**What does cleaning involve?**
1. Extract just the Adjusted Close prices (that's what we use for calculations)
2. Handle missing data
3. Calculate daily returns (how much each stock went up/down each day)

In [ ]:
# MEMBER A: Clean the data

# Step 1: Extract Adjusted Close prices
prices = raw_data['Adj Close']

# Step 2: Handle missing values
# dropna(how='all') = only drop a row if ALL stocks are missing that day
# ffill() = "forward fill" — if a value is missing, use yesterday's value
prices = prices.dropna(how='all').ffill()

# Check for remaining missing values
missing = prices.isnull().sum()
if missing.any():
    print('WARNING: Still some missing values:')
    print(missing[missing > 0])
    prices = prices.bfill()  # backward fill any remaining gaps
else:
    print('No missing values — data is clean!')

# Step 3: Separate stocks from benchmark
stock_prices = prices[TICKERS]
benchmark_prices = prices[BENCHMARK]

# Step 4: Calculate daily returns
# Return = (today's price - yesterday's price) / yesterday's price
# Example: price goes $100 → $103, return = (103-100)/100 = 0.03 = 3%
stock_returns = stock_prices.pct_change().dropna()
benchmark_returns = benchmark_prices.pct_change().dropna()

# Free memory
del raw_data

print(f'\nClean price data: {stock_prices.shape}')
print(f'Daily returns: {stock_returns.shape}')
print(f'\nFirst 3 rows of returns:')
stock_returns.head(3)

In [ ]:
# MEMBER A: Quick sanity check — does the data look reasonable?

print('SANITY CHECK')
print('=' * 60)

# Check 1: Are returns reasonable? (daily returns > 50% are suspicious)
max_return = stock_returns.max().max() * 100
min_return = stock_returns.min().min() * 100
print(f'Largest daily gain:  {max_return:.1f}% — {"OK" if max_return < 50 else "SUSPICIOUS!"}')
print(f'Largest daily loss:  {min_return:.1f}% — {"OK" if min_return > -50 else "SUSPICIOUS!"}')

# Check 2: Do we have enough data?
expected_days = 252 * 5  # ~252 trading days per year × 5 years
actual_days = len(stock_returns)
print(f'Trading days: {actual_days} (expected ~{expected_days}) — {"OK" if actual_days > 1000 else "LOW!"}')

# Check 3: Any stock with constant price? (that would be wrong)
zero_variance = (stock_returns.std() == 0).any()
print(f'Zero-variance stocks: {"NONE (good!)" if not zero_variance else "PROBLEM!"}')

print('\nIf everything says OK, your data is ready for Member B!')

## Task A4: Calculate Technical Indicators

**What are technical indicators?**
They are math formulas applied to stock prices that help predict future movements.

Think of them as **thermometers for the stock market**:

| Indicator | What It Tells You | Analogy |
|-----------|-------------------|----------|
| **SMA** (Simple Moving Average) | Is the stock trending up or down? | Like a smoothed-out trend line |
| **RSI** (Relative Strength Index) | Is the stock overbought or oversold? (0-100) | Like a "too hot / too cold" meter |
| **MACD** | Is momentum increasing or decreasing? | Like checking if a car is speeding up or slowing down |
| **Bollinger Bands** | Is the price unusually high or low? | Like checking if temperature is normal or extreme |

**Member B will use these indicators** as inputs to their trading strategy.

In [ ]:
import ta  # Technical Analysis library — does all the math for us

def calculate_indicators(price_series):
    """
    Calculate technical indicators for one stock.
    
    Input:  a pandas Series of daily closing prices (e.g., AAPL prices)
    Output: a DataFrame with the price + all indicators as columns
    """
    df = pd.DataFrame({'Close': price_series})
    
    # --- Moving Averages ---
    # SMA = average of last N days' prices
    # If price is ABOVE the SMA → stock is in an uptrend
    # If price is BELOW the SMA → stock is in a downtrend
    df['SMA_20']  = df['Close'].rolling(window=20).mean()   # 20-day average
    df['SMA_50']  = df['Close'].rolling(window=50).mean()   # 50-day average
    df['SMA_200'] = df['Close'].rolling(window=200).mean()  # 200-day average
    
    # --- RSI (Relative Strength Index) ---
    # Scale: 0 to 100
    # Below 30 = "oversold" (stock fell too much → might bounce back = BUY signal)
    # Above 70 = "overbought" (stock rose too much → might drop = SELL signal)
    df['RSI'] = ta.momentum.RSIIndicator(df['Close'], window=14).rsi()
    
    # --- MACD (Moving Average Convergence Divergence) ---
    # Measures momentum (speed of price change)
    # When MACD > Signal line → bullish (price likely to go up)
    # When MACD < Signal line → bearish (price likely to go down)
    macd = ta.trend.MACD(df['Close'])
    df['MACD'] = macd.macd()
    df['MACD_Signal'] = macd.macd_signal()
    df['MACD_Hist'] = macd.macd_diff()
    
    # --- Bollinger Bands ---
    # Creates a band around the price
    # Upper band = unusually HIGH price
    # Lower band = unusually LOW price
    bb = ta.volatility.BollingerBands(df['Close'])
    df['BB_Upper'] = bb.bollinger_hband()
    df['BB_Lower'] = bb.bollinger_lband()
    df['BB_Mid']   = bb.bollinger_mavg()
    
    # --- Volatility ---
    # How much the price bounces around (higher = riskier)
    df['Daily_Return'] = df['Close'].pct_change()
    df['Volatility_20'] = df['Daily_Return'].rolling(window=20).std() * np.sqrt(252)
    
    return df

# Calculate indicators for ALL stocks
indicators_dict = {}
for ticker in TICKERS:
    indicators_dict[ticker] = calculate_indicators(stock_prices[ticker])
    
print(f'Calculated indicators for {len(indicators_dict)} stocks.')
print(f'\nExample — AAPL indicators (last 3 rows):')
indicators_dict['AAPL'].tail(3)

## Task A5: Prepare Data for Member B

Member B needs the data in specific formats depending on which strategy they're building.

**For the RL strategy (FinRL):** Data needs to be in "long format" — one row per stock per day.

**For the LLM strategy:** Data is already in the right format (our `indicators_dict`).

**What to do:** Run the cell below to prepare both formats.

In [ ]:
# MEMBER A: Prepare data for FinRL (if Member B chooses the RL path)
#
# FinRL needs data in "long format":
# | date       | tic  | close | high | low  | open | volume | macd | rsi |
# | 2020-01-02 | AAPL | 75.0  | 75.5 | 74.5 | 74.8 | 1000   | 0.5  | 45  |
# | 2020-01-02 | MSFT | 157.0 | ...  | ...  | ...  | ...    | ...  | ... |
#
# Our current format is "wide" (one column per stock)
# We need to convert it

try:
    from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
    from finrl.meta.preprocessor.preprocessors import FeatureEngineer
    
    print('Downloading data in FinRL format...')
    df_finrl = YahooDownloader(
        start_date=START_DATE,
        end_date=END_DATE,
        ticker_list=TICKERS
    ).fetch_data()
    
    # Add technical indicators
    FINRL_INDICATORS = ['macd', 'rsi_30', 'cci_30', 'dx_30']
    fe = FeatureEngineer(
        use_technical_indicator=True,
        tech_indicator_list=FINRL_INDICATORS,
        use_turbulence=True,
        user_defined_feature=False
    )
    df_finrl_processed = fe.preprocess_data(df_finrl)
    df_finrl_processed = df_finrl_processed.sort_values(['date', 'tic']).reset_index(drop=True)
    
    # Split into train/test
    train_df_finrl = df_finrl_processed[df_finrl_processed['date'] < SPLIT_DATE]
    test_df_finrl = df_finrl_processed[df_finrl_processed['date'] >= SPLIT_DATE]
    
    print(f'FinRL data ready!')
    print(f'  Training: {len(train_df_finrl)} rows')
    print(f'  Testing:  {len(test_df_finrl)} rows')
    
except ImportError:
    print('FinRL not installed. Run: pip install finrl')
    print('This is only needed if Member B chooses the RL path.')

In [ ]:
# MEMBER A: Split the main data into train/test sets for Member B
# 
# WHY split?
# Training data = what we use to BUILD the strategy (like studying for an exam)
# Testing data  = what we use to EVALUATE the strategy (like taking the exam)
# If you test on the same data you trained on, you're cheating!

train_returns = stock_returns[stock_returns.index < SPLIT_DATE]
test_returns  = stock_returns[stock_returns.index >= SPLIT_DATE]

train_prices = stock_prices[stock_prices.index < SPLIT_DATE]
test_prices  = stock_prices[stock_prices.index >= SPLIT_DATE]

benchmark_test_returns = benchmark_returns[benchmark_returns.index >= SPLIT_DATE]

print('DATA READY FOR MEMBER B')
print('=' * 50)
print(f'Training: {train_returns.index[0].date()} to {train_returns.index[-1].date()} ({len(train_returns)} days)')
print(f'Testing:  {test_returns.index[0].date()} to {test_returns.index[-1].date()} ({len(test_returns)} days)')
print(f'\nVariables Member B should use:')
print(f'  stock_prices      — daily prices (all dates)')
print(f'  stock_returns     — daily returns (all dates)')
print(f'  train_returns     — daily returns (training period only)')
print(f'  test_returns      — daily returns (testing period only)')
print(f'  indicators_dict   — technical indicators per stock')
print(f'  benchmark_returns — SPY daily returns')
print(f'  INITIAL_CAPITAL   — ${INITIAL_CAPITAL:,}')

## Task A6: Set Up Alpaca Paper Trading

**What is Alpaca?**
Alpaca is a platform that lets you trade stocks through code. "Paper trading" means using **fake money** to test strategies in the real market.

**Step-by-step setup:**

1. Go to https://app.alpaca.markets/signup
2. Create a free account (email + password)
3. After logging in, click **"Paper Trading"** in the left sidebar
4. Click **"View API Keys"** → **"Generate New Key"**
5. You'll see two values:
   - **API Key ID**: something like `PKxxxxxxxxxxxxxxxx`
   - **Secret Key**: a longer string (SAVE THIS — you can't see it again!)
6. Set them as environment variables (see code below)

**IMPORTANT: Never put your API keys directly in the notebook!** If you share the notebook, anyone can use your account.

In [ ]:
# MEMBER A: Set up Alpaca connection

import os
from alpaca_trade_api import REST

# Option 1: Set environment variables in your terminal BEFORE opening Jupyter:
#   export ALPACA_API_KEY='PKxxxxxxxxxxxxxxxx'
#   export ALPACA_SECRET_KEY='xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'
#
# Option 2 (Windows): Set in System Settings → Environment Variables
#
# Option 3 (temporary, for testing only): Uncomment and paste below
#   os.environ['ALPACA_API_KEY'] = 'PKxxxxxxxxxxxxxxxx'
#   os.environ['ALPACA_SECRET_KEY'] = 'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'

ALPACA_API_KEY = os.environ.get('ALPACA_API_KEY', '')
ALPACA_SECRET_KEY = os.environ.get('ALPACA_SECRET_KEY', '')
ALPACA_BASE_URL = 'https://paper-api.alpaca.markets'  # This is PAPER trading (fake money)

if not ALPACA_API_KEY or not ALPACA_SECRET_KEY:
    print('Alpaca API keys not found!')
    print('Follow the steps above to get your keys.')
    alpaca = None
else:
    alpaca = REST(ALPACA_API_KEY, ALPACA_SECRET_KEY, ALPACA_BASE_URL, api_version='v2')
    try:
        account = alpaca.get_account()
        print('Connected to Alpaca Paper Trading!')
        print(f'Cash available: ${float(account.cash):,.2f}')
        print(f'Portfolio value: ${float(account.portfolio_value):,.2f}')
    except Exception as e:
        print(f'Connection failed: {e}')
        alpaca = None

In [ ]:
# MEMBER A: Function to execute trades on Alpaca
# Member B will call this function with their strategy's weights

def rebalance_portfolio(target_weights):
    """
    Rebalance the Alpaca portfolio to match target weights.
    
    Input: dict like {'AAPL': 0.15, 'MSFT': 0.10, ...}
           where values are the fraction of portfolio to allocate (0.15 = 15%)
    
    What it does:
    1. Checks current positions
    2. Calculates how much to buy/sell to reach target
    3. Places market orders
    """
    if alpaca is None:
        print('Alpaca not connected. Set up API keys first.')
        return []
    
    account = alpaca.get_account()
    portfolio_value = float(account.portfolio_value)
    positions = {p.symbol: float(p.market_value) for p in alpaca.list_positions()}
    
    orders = []
    for ticker in TICKERS:
        target_value = portfolio_value * target_weights.get(ticker, 0)
        current_value = positions.get(ticker, 0)
        diff = target_value - current_value
        
        if abs(diff) < 100:  # Skip tiny differences
            continue
        
        try:
            price = alpaca.get_latest_trade(ticker).price
        except Exception as e:
            print(f'  Could not get price for {ticker}: {e}')
            continue
        
        shares = int(abs(diff) / price)
        if shares == 0:
            continue
        
        side = 'buy' if diff > 0 else 'sell'
        try:
            alpaca.submit_order(symbol=ticker, qty=shares, side=side,
                               type='market', time_in_force='day')
            orders.append(f'{side.upper()} {shares} {ticker}')
            print(f'  {side.upper()} {shares} shares of {ticker}')
        except Exception as e:
            print(f'  Failed {side} {ticker}: {e}')
    
    print(f'\nPlaced {len(orders)} orders.')
    return orders

print('rebalance_portfolio() function ready for Member B to use.')

## Task A7: Notebook Organization (Final Step)

**When to do this:** At the very end, after Members B and C are done.

**Checklist:**
- [ ] All cells run in order from top to bottom without errors
- [ ] All cells have clear comments explaining what they do
- [ ] Markdown headers separate each section
- [ ] No API keys are visible in the notebook
- [ ] No unnecessary print statements or debug output
- [ ] Restart kernel and "Run All" to verify everything works

---
---
# MEMBER B: Strategy Development & Backtesting
---
---

## Your Role in Plain English

You are the **chef**. Member A gave you clean ingredients (data). Your job is to create the recipe (strategy) that turns data into trading decisions.

## Your Tasks (in order):

1. Build **Benchmark Strategy 1**: Buy & Hold S&P 500
2. Build **Benchmark Strategy 2**: Mean-Variance Optimization
3. Build **Main Strategy** (choose one or both):
   - Path A: Reinforcement Learning with FinRL
   - Path B: LLM-based Trading Signals
4. Backtest all strategies on test data
5. Send results to Member C for analysis

## What does "strategy" even mean?

A trading strategy is just a **set of rules** that answers:
> "Given today's data, how should I split my money across stocks?"

The output is always a set of **weights** — what percentage goes to each stock:
```
{'AAPL': 0.15, 'MSFT': 0.10, 'GOOGL': 0.08, ...}  # must sum to ~1.0
```

## Task B1: Benchmark — Buy & Hold S&P 500

**What is this?**
The simplest possible strategy: buy the S&P 500 on day 1 and never sell.

**Why do we need this?**
If your fancy algorithm can't beat someone who just buys SPY and does nothing, your algorithm is useless. This is the **minimum bar** to clear.

**What is the S&P 500?**
It's an index that tracks the 500 largest US companies. SPY is an ETF (fund) that mirrors it. When people say "the market is up 2%", they usually mean the S&P 500.

In [ ]:
# MEMBER B: Performance metrics function
# You'll use this to evaluate EVERY strategy
# Run this cell first — all later cells depend on it

def calculate_performance(returns, name='Strategy'):
    """
    Calculate trading performance metrics from daily returns.
    
    Input:  pd.Series of daily returns (e.g., 0.02 means +2% that day)
    Output: dict of performance metrics
    
    Key metrics explained:
    - Total Return: how much money you made overall
    - Annual Return: return per year (annualized)
    - Annual Volatility: how much the portfolio bounces around per year (risk)
    - Sharpe Ratio: return per unit of risk (higher = better, >1 is good)
    - Max Drawdown: worst peak-to-trough decline (how bad it got)
    - Win Rate: what % of days were profitable
    """
    total_return = (1 + returns).prod() - 1
    n_years = len(returns) / 252  # 252 trading days per year
    annual_return = (1 + total_return) ** (1 / n_years) - 1
    annual_vol = returns.std() * np.sqrt(252)
    
    risk_free_rate = 0.04  # 4% — approximate US Treasury yield
    sharpe = (annual_return - risk_free_rate) / annual_vol
    
    # Max drawdown
    cumulative = (1 + returns).cumprod()
    max_drawdown = (cumulative / cumulative.cummax() - 1).min()
    
    # Sortino ratio (like Sharpe but only penalizes losses, not gains)
    downside_vol = returns[returns < 0].std() * np.sqrt(252)
    sortino = (annual_return - risk_free_rate) / downside_vol if downside_vol > 0 else 0
    
    return {
        'Strategy': name,
        'Total Return (%)': round(total_return * 100, 2),
        'Annual Return (%)': round(annual_return * 100, 2),
        'Annual Volatility (%)': round(annual_vol * 100, 2),
        'Sharpe Ratio': round(sharpe, 3),
        'Sortino Ratio': round(sortino, 3),
        'Max Drawdown (%)': round(max_drawdown * 100, 2),
        'Win Rate (%)': round((returns > 0).mean() * 100, 2),
    }

print('calculate_performance() function ready.')
print('Use it like: metrics = calculate_performance(some_returns, "My Strategy")')

In [ ]:
# MEMBER B: Buy & Hold S&P 500 benchmark

# This is trivial — just use the benchmark returns directly
# The "strategy" is: put 100% in SPY on day 1, never touch it

bh_metrics = calculate_performance(benchmark_test_returns, 'Buy & Hold SPY')

# Calculate portfolio value over time (for plotting later)
bh_portfolio = INITIAL_CAPITAL * (1 + benchmark_test_returns).cumprod()

print('BENCHMARK 1: Buy & Hold S&P 500')
print('=' * 50)
for k, v in bh_metrics.items():
    if k != 'Strategy':
        print(f'  {k}: {v}')
print(f'\nFinal portfolio value: ${bh_portfolio.iloc[-1]:,.0f}')

## Task B2: Benchmark — Mean-Variance Optimization (MVO)

**This is a REQUIRED benchmark** per the project instructions.

### The Big Idea (Markowitz Portfolio Theory)

Imagine you have $100 and 15 stocks. How do you split the money?

**Naive approach:** Put equal money in each stock ($6.67 each). But some stocks are riskier than others!

**Smart approach (MVO):** Find the mix that gives the **best return for a given level of risk**.

### How MVO Works (Step by Step)

1. **Calculate expected return** for each stock (use historical average)
2. **Calculate the covariance matrix** (how stocks move together)
3. **Optimize**: find weights that maximize the Sharpe Ratio

### Key Concept: Covariance

- If AAPL goes up and MSFT also goes up → **high covariance** (they move together)
- If AAPL goes up and XOM goes down → **low/negative covariance** (they move opposite)
- **Low covariance = good for diversification!** When one drops, the other rises.

### Key Concept: Sharpe Ratio

```
Sharpe Ratio = (Return - Risk-Free Rate) / Risk
```

A strategy with 20% return and 10% volatility has Sharpe = (20-4)/10 = 1.6

A strategy with 20% return and 40% volatility has Sharpe = (20-4)/40 = 0.4

**Higher Sharpe = better risk-adjusted performance.** Above 1.0 is considered good.

In [ ]:
# MEMBER B: Mean-Variance Optimization

from scipy.optimize import minimize

# Step 1: Calculate inputs from TRAINING data only
# (We NEVER use test data to build our strategy — that would be cheating)

expected_returns = train_returns.mean() * 252  # Annualized expected returns
cov_matrix = train_returns.cov() * 252         # Annualized covariance matrix
n_assets = len(TICKERS)

print('Expected Annual Returns (from training data):')
for ticker in TICKERS:
    print(f'  {ticker}: {expected_returns[ticker]*100:.1f}%')

In [ ]:
# MEMBER B: Define optimization helper functions

def portfolio_return(weights):
    """Portfolio return = weighted average of stock returns."""
    return np.dot(weights, expected_returns)

def portfolio_volatility(weights):
    """Portfolio risk. NOT just weighted average — correlations matter!"""
    return np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))

def neg_sharpe(weights):
    """Negative Sharpe Ratio. We MINIMIZE this (= MAXIMIZE Sharpe)."""
    ret = portfolio_return(weights)
    vol = portfolio_volatility(weights)
    return -(ret - 0.04) / vol

# Constraints:
# 1. Weights must sum to 1 (invest all money)
# 2. Each weight between 0 and 1 (no short selling, no leverage)
constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
bounds = tuple((0, 1) for _ in range(n_assets))
initial = np.array([1/n_assets] * n_assets)  # Start with equal weights

print('Optimization functions ready.')

In [ ]:
# MEMBER B: Run the optimization

# Find Maximum Sharpe Ratio portfolio
result = minimize(neg_sharpe, initial, method='SLSQP', bounds=bounds, constraints=constraints)
mvo_weights = result.x

# Find Minimum Volatility portfolio
result_mv = minimize(portfolio_volatility, initial, method='SLSQP', bounds=bounds, constraints=constraints)
minvol_weights = result_mv.x

# Equal weight (another simple benchmark)
equal_weights = np.array([1/n_assets] * n_assets)

# Display results
print('OPTIMAL PORTFOLIO WEIGHTS')
print('=' * 60)
print(f'{"Stock":<8} {"Max Sharpe":<15} {"Min Vol":<15} {"Equal":<15}')
print('-' * 60)
for i, ticker in enumerate(TICKERS):
    if mvo_weights[i] > 0.005 or minvol_weights[i] > 0.005:
        print(f'{ticker:<8} {mvo_weights[i]*100:>10.1f}%    {minvol_weights[i]*100:>10.1f}%    {equal_weights[i]*100:>10.1f}%')

print(f'\nMax Sharpe — Expected Return: {portfolio_return(mvo_weights)*100:.1f}%, Vol: {portfolio_volatility(mvo_weights)*100:.1f}%')
print(f'Min Vol    — Expected Return: {portfolio_return(minvol_weights)*100:.1f}%, Vol: {portfolio_volatility(minvol_weights)*100:.1f}%')

In [ ]:
# MEMBER B: Backtest MVO on TEST data

# Apply the weights (found from training) to test period returns
mvo_test_returns = test_returns @ mvo_weights  # @ = matrix multiply
minvol_test_returns = test_returns @ minvol_weights
equal_test_returns = test_returns @ equal_weights

# Align with benchmark dates
common_idx = mvo_test_returns.index.intersection(benchmark_test_returns.index)
mvo_test_returns = mvo_test_returns.loc[common_idx]
minvol_test_returns = minvol_test_returns.loc[common_idx]
equal_test_returns = equal_test_returns.loc[common_idx]

# Calculate portfolio values
mvo_portfolio = INITIAL_CAPITAL * (1 + mvo_test_returns).cumprod()
minvol_portfolio = INITIAL_CAPITAL * (1 + minvol_test_returns).cumprod()
equal_portfolio = INITIAL_CAPITAL * (1 + equal_test_returns).cumprod()

# Calculate metrics
mvo_metrics = calculate_performance(mvo_test_returns, 'MVO (Max Sharpe)')
minvol_metrics = calculate_performance(minvol_test_returns, 'MVO (Min Vol)')
equal_metrics = calculate_performance(equal_test_returns, 'Equal Weight')

benchmark_comparison = pd.DataFrame([bh_metrics, mvo_metrics, minvol_metrics, equal_metrics]).set_index('Strategy')
print('BENCHMARK STRATEGIES — TEST PERIOD')
print('=' * 80)
benchmark_comparison

## Task B3: Main Strategy — Choose Your Path

Now it's time to build your **main strategy**. Choose one (or both!) paths:

### Path A: Reinforcement Learning (FinRL)
- **Difficulty**: Higher (needs ML knowledge)
- **How it works**: A computer agent "plays" the market millions of times and learns which actions lead to profit
- **Analogy**: Teaching a robot to play a video game by trial and error

### Path B: LLM-Based Trading Signals
- **Difficulty**: Lower (mostly prompt engineering)
- **How it works**: Feed market data to ChatGPT/Claude and ask it to make trading decisions
- **Analogy**: Hiring an AI analyst who reads the data and gives you trading advice

**Scroll down to the path you chose.**

### PATH A: Reinforcement Learning with FinRL

**What is Reinforcement Learning?**

Imagine teaching a dog tricks:
1. Dog does something → good result → give treat (positive reward)
2. Dog does something → bad result → no treat (negative reward)
3. Over time, the dog learns which actions get treats

RL works the same way but with a computer:
```
Agent sees market data (state)
  → Decides to BUY/SELL/HOLD (action)
  → Portfolio goes up or down (reward)
  → Agent remembers what worked
  → Repeat 50,000 times
```

**Algorithms we'll try:**
- **PPO**: Most popular, very stable
- **A2C**: Fast training, simpler
- **DDPG**: Good for deciding exact amounts to buy/sell

In [ ]:
# MEMBER B (PATH A): Set up FinRL environment

from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent

# Environment configuration
# The "environment" is the simulated market the agent trades in
FINRL_INDICATORS = ['macd', 'rsi_30', 'cci_30', 'dx_30']
stock_dimension = len(TICKERS)
state_space = 1 + 2 * stock_dimension + len(FINRL_INDICATORS) * stock_dimension

env_kwargs = {
    'hmax': 100,                          # Max shares per trade
    'initial_amount': INITIAL_CAPITAL,     # Starting cash
    'buy_cost_pct': [0.001] * stock_dimension,   # 0.1% transaction fee
    'sell_cost_pct': [0.001] * stock_dimension,  # 0.1% transaction fee
    'state_space': state_space,
    'stock_dim': stock_dimension,
    'tech_indicator_list': FINRL_INDICATORS,
    'action_space': stock_dimension,
    'reward_scaling': 1e-4
}

# Create training environment
e_train = StockTradingEnv(df=train_df_finrl, **env_kwargs)
env_train, _ = e_train.get_sb_env()

print(f'Environment ready!')
print(f'State space: {state_space} dimensions')
print(f'Action space: {stock_dimension} (one decision per stock)')

In [ ]:
# MEMBER B (PATH A): Train all 3 RL agents
# This takes 5-15 minutes depending on your computer

TIMESTEPS = 50000  # Training steps (more = better but slower)

AGENT_CONFIGS = [
    ('ppo',  {'n_steps': 2048, 'learning_rate': 0.0003}),
    ('a2c',  {'n_steps': 5,    'learning_rate': 0.0007}),
    ('ddpg', {'batch_size': 128, 'learning_rate': 0.001}),
]

trained_models = {}
for algo_name, kwargs in AGENT_CONFIGS:
    print(f'\nTraining {algo_name.upper()}...')
    agent = DRLAgent(env=env_train)
    model = agent.get_model(algo_name, model_kwargs=kwargs)
    trained_models[algo_name] = agent.train_model(
        model=model, tb_log_name=algo_name, total_timesteps=TIMESTEPS
    )
    print(f'{algo_name.upper()} done!')

print('\nAll agents trained!')

In [ ]:
# MEMBER B (PATH A): Test all agents on unseen data

rl_results = {}
for algo_name, model in trained_models.items():
    print(f'Testing {algo_name.upper()}...')
    test_env = StockTradingEnv(df=test_df_finrl, **env_kwargs)
    df_account, df_actions = DRLAgent.DRL_prediction(model=model, environment=test_env)
    rl_results[algo_name] = df_account

# Convert to returns and calculate metrics
rl_metrics_list = []
for algo_name, df_acc in rl_results.items():
    values = df_acc['account_value']
    rets = values.pct_change().dropna()
    rets.index = pd.to_datetime(df_acc['date'].iloc[1:].values)
    metrics = calculate_performance(rets, f'RL — {algo_name.upper()}')
    rl_metrics_list.append(metrics)
    print(f'{algo_name.upper()} final value: ${values.iloc[-1]:,.0f}')

pd.DataFrame(rl_metrics_list).set_index('Strategy')

### PATH B: LLM-Based Trading Signals

**How this works:**
1. We gather today's market data (prices, RSI, MACD, etc.)
2. We write a **prompt** — instructions telling the LLM what to analyze
3. We send it to GPT-4 / Claude via API
4. The LLM responds with BUY/SELL/HOLD signals for each stock
5. We convert those signals into portfolio weights
6. Repeat every week

**What is an API?**
API = Application Programming Interface. It's a way for code to talk to another service. Instead of going to ChatGPT's website and typing, we send the message through code.

**What is a prompt?**
The instructions you write for the LLM. A good prompt = good results. This is called **prompt engineering**.

**Setup:** You need an OpenAI API key.
1. Go to https://platform.openai.com/api-keys
2. Create a key
3. Set it: `export OPENAI_API_KEY='sk-...'`

In [ ]:
# MEMBER B (PATH B): Set up LLM connection

import json
from openai import OpenAI

OPENAI_KEY = os.environ.get('OPENAI_API_KEY', '')
if not OPENAI_KEY:
    print('WARNING: OPENAI_API_KEY not set!')
    print('LLM backtest will use equal weights as fallback.')
    llm_client = None
else:
    llm_client = OpenAI(api_key=OPENAI_KEY)
    print('OpenAI client connected!')

LLM_MODEL = 'gpt-4o-mini'  # Cheapest option (~$0.15 per 1M tokens)
print(f'Model: {LLM_MODEL}')

In [ ]:
# MEMBER B (PATH B): Build the prompt
# This is the most important part — the quality of your prompt determines
# the quality of the LLM's trading decisions

def build_prompt(date, tickers, prices_df, ind_dict, lookback=10):
    """
    Build a prompt with market data for the LLM.
    
    The prompt gives the LLM:
    - Current price of each stock
    - Recent price changes
    - Technical indicator values
    - Instructions on how to respond
    """
    recent = prices_df[prices_df.index <= date].tail(lookback)
    
    stock_info = ""
    for ticker in tickers:
        ind = ind_dict[ticker]
        row = ind[ind.index <= date].tail(1)
        if len(row) == 0:
            continue
        latest = row.iloc[-1]
        
        # Calculate recent price changes
        p = recent[ticker] if ticker in recent.columns else None
        chg_5d = (p.iloc[-1] / p.iloc[-5] - 1) * 100 if p is not None and len(p) >= 5 else 0
        chg_10d = (p.iloc[-1] / p.iloc[0] - 1) * 100 if p is not None and len(p) >= 10 else 0
        
        rsi_val = latest.get('RSI', 50)
        rsi_note = '(OVERSOLD)' if rsi_val < 30 else '(OVERBOUGHT)' if rsi_val > 70 else ''
        macd_note = 'BULLISH' if latest.get('MACD', 0) > latest.get('MACD_Signal', 0) else 'BEARISH'
        
        stock_info += f"""\n{ticker}: Price=${latest['Close']:.2f} | 5d={chg_5d:+.1f}% | 10d={chg_10d:+.1f}% | RSI={rsi_val:.0f}{rsi_note} | MACD={macd_note} | Vol={latest.get('Volatility_20',0)*100:.0f}%"""
    
    prompt = f"""You are a portfolio manager. Analyze this data for {date.strftime('%Y-%m-%d')} and allocate a portfolio.

RULES:
- Allocate 0-20% per stock, total ~100%
- Favor stocks with RSI < 30 (oversold = potential bounce)
- Avoid stocks with RSI > 70 (overbought = potential drop)
- Favor BULLISH MACD momentum
- Diversify across sectors

DATA:
{stock_info}

Respond with ONLY valid JSON:
{{"signals": {{"TICKER": {{"action": "BUY"|"SELL"|"HOLD", "weight": 0.0-0.20}}, ...}}, "reasoning": "brief explanation"}}"""
    return prompt

# Preview
sample = build_prompt(stock_prices.index[300], TICKERS, stock_prices, indicators_dict)
print(sample[:600] + '\n...')

In [ ]:
# MEMBER B (PATH B): LLM calling and signal parsing functions

def call_llm(prompt):
    """Send prompt to LLM, return parsed JSON or None on failure."""
    if llm_client is None:
        return None
    try:
        response = llm_client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": "You are a portfolio manager. Respond with valid JSON only."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3,
            max_tokens=1000
        )
        text = response.choices[0].message.content.strip()
        if text.startswith('```'):
            text = text.split('```')[1]
            if text.startswith('json'):
                text = text[4:]
        return json.loads(text)
    except Exception as e:
        print(f'  LLM error: {e}')
        return None

def parse_signals(signals, tickers):
    """Convert LLM signals to portfolio weights dict."""
    if signals is None or 'signals' not in signals:
        return {t: 1/len(tickers) for t in tickers}  # Equal weight fallback
    
    weights = {}
    for t in tickers:
        if t in signals['signals']:
            sig = signals['signals'][t]
            action = sig.get('action', 'HOLD').upper()
            w = min(float(sig.get('weight', 0.05)), 0.20)
            weights[t] = 0.0 if action == 'SELL' else w
        else:
            weights[t] = 0.0
    
    total = sum(weights.values())
    if total > 0:
        return {k: v/total for k, v in weights.items()}
    return {t: 1/len(tickers) for t in tickers}

print('LLM functions ready.')

In [ ]:
# MEMBER B (PATH B): Run the LLM backtest
# Calls the LLM every 5 trading days to get new portfolio weights

REBALANCE_DAYS = 5  # Rebalance weekly
test_dates = stock_prices[stock_prices.index >= SPLIT_DATE].index
test_stock_rets = stock_returns.reindex(test_dates).fillna(0)

llm_values = [INITIAL_CAPITAL]
llm_failed = 0
weights = {t: 1/len(TICKERS) for t in TICKERS}

print(f'Running LLM backtest: {len(test_dates)} days, ~{len(test_dates)//REBALANCE_DAYS} API calls\n')

for i in range(1, len(test_dates)):
    date = test_dates[i]
    
    if i % REBALANCE_DAYS == 0:
        signals = call_llm(build_prompt(date, TICKERS, stock_prices, indicators_dict))
        if signals is None:
            llm_failed += 1
        weights = parse_signals(signals, TICKERS)
        if i % 25 == 0:
            print(f'  Day {i}/{len(test_dates)} — {date.strftime("%Y-%m-%d")} — ${llm_values[-1]:,.0f}')
    
    # Daily return = weighted sum of stock returns
    w = np.array([weights[t] for t in TICKERS])
    daily_ret = np.dot(w, test_stock_rets.loc[date][TICKERS].values)
    llm_values.append(llm_values[-1] * (1 + daily_ret))

# Build returns series
llm_portfolio = pd.Series(llm_values[1:], index=test_dates[:len(llm_values)-1])
llm_returns = llm_portfolio.pct_change().dropna()
llm_metrics = calculate_performance(llm_returns, f'LLM ({LLM_MODEL})')

total_calls = len(test_dates) // REBALANCE_DAYS
print(f'\nDone! Failed calls: {llm_failed}/{total_calls} ({llm_failed/max(total_calls,1)*100:.0f}%)')
print(f'Final value: ${llm_values[-1]:,.0f}')
pd.DataFrame([llm_metrics]).set_index('Strategy')

## Task B4: Send Results to Member C

Member C needs the following variables from you. Make sure they're all defined:

**Benchmark results:**
- `bh_metrics`, `bh_portfolio` — Buy & Hold SPY
- `mvo_metrics`, `mvo_portfolio`, `mvo_test_returns` — MVO Max Sharpe
- `minvol_metrics`, `minvol_portfolio`, `minvol_test_returns` — MVO Min Vol
- `equal_metrics`, `equal_portfolio`, `equal_test_returns` — Equal Weight

**Main strategy results:**
- Path A: `rl_results`, `rl_metrics_list`
- Path B: `llm_metrics`, `llm_portfolio`, `llm_returns`

**Shared:**
- `benchmark_test_returns` — SPY test returns
- `calculate_performance()` — the metrics function

In [ ]:
# MEMBER B: Quick check — are all results ready for Member C?

print('HANDOFF CHECK FOR MEMBER C')
print('=' * 50)

required = {
    'bh_metrics': 'Buy & Hold metrics',
    'mvo_metrics': 'MVO metrics',
    'benchmark_test_returns': 'SPY test returns',
    'mvo_test_returns': 'MVO test returns',
}

optional = {
    'rl_metrics_list': 'RL metrics (Path A)',
    'llm_metrics': 'LLM metrics (Path B)',
}

for var, desc in required.items():
    status = 'READY' if var in dir() else 'MISSING!'
    print(f'  [{status}] {desc} ({var})')

print('\nOptional (depends on chosen path):')
for var, desc in optional.items():
    status = 'READY' if var in dir() else 'not done'
    print(f'  [{status}] {desc} ({var})')

---
---
# MEMBER C: Analysis & Reporting
---
---

## Your Role in Plain English

You are the **food critic and storyteller**. Member A gathered ingredients, Member B cooked the meal. You:
1. **Analyze** how well each strategy performed
2. **Visualize** the results with charts
3. **Assess risk** (how dangerous is each strategy?)
4. **Write the report** that explains everything

## Your Tasks (in order):

1. Create the **EDA visualizations** (exploratory data analysis charts)
2. Build the **grand comparison table and dashboard**
3. Conduct **risk analysis** (drawdowns, VaR, rolling Sharpe)
4. Create the **Efficient Frontier** plot
5. Write the **20-page PDF report**
6. Write the **5-page paper trading follow-up report**

## Task C1: EDA Visualizations

**EDA = Exploratory Data Analysis**

Before showing strategy results, we need to show the reader what our data looks like. These charts go in the "Introduction" section of the report.

Charts to create:
1. **Normalized price chart** — how each stock performed over time
2. **Risk vs Return scatter** — which stocks are best?
3. **Correlation heatmap** — how stocks move together
4. **Technical indicator example** — what RSI/MACD look like

In [ ]:
# MEMBER C: Set up plot style
import seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12

print('Plot style configured.')

In [ ]:
# MEMBER C: Chart 1 — Normalized Price Chart
# This shows how $1 invested in each stock would have grown

norm_prices = stock_prices / stock_prices.iloc[0]  # Divide by first day's price
norm_benchmark = benchmark_prices / benchmark_prices.iloc[0]

fig, ax = plt.subplots(figsize=(16, 8))
norm_prices.plot(ax=ax, alpha=0.7)
norm_benchmark.plot(ax=ax, color='black', linewidth=3, linestyle='--', label='SPY (Benchmark)')

ax.set_title('Normalized Stock Prices (Growth of $1)', fontsize=16)
ax.set_ylabel('Growth of $1')
ax.set_xlabel('Date')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('chart_normalized_prices.png', dpi=150, bbox_inches='tight')  # Save for report
plt.show()
print('Saved: chart_normalized_prices.png')

In [ ]:
# MEMBER C: Chart 2 — Risk vs Return Scatter
# X = risk (volatility), Y = return
# Top-left = best (high return, low risk)

ann_ret = stock_returns.mean() * 252
ann_vol = stock_returns.std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(ann_vol * 100, ann_ret * 100, s=100, alpha=0.7, edgecolors='black')

for ticker in TICKERS:
    ax.annotate(ticker, (ann_vol[ticker]*100, ann_ret[ticker]*100), fontsize=10, ha='center', va='bottom')

# Add benchmark
bm_r = benchmark_returns.mean() * 252 * 100
bm_v = benchmark_returns.std() * np.sqrt(252) * 100
ax.scatter(bm_v, bm_r, s=200, color='red', marker='*', zorder=5)
ax.annotate('SPY', (bm_v, bm_r), fontsize=12, fontweight='bold', color='red')

ax.set_title('Risk vs Return (Annualized)', fontsize=16)
ax.set_xlabel('Annual Volatility (Risk) %')
ax.set_ylabel('Annual Return %')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('chart_risk_return.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart_risk_return.png')

In [ ]:
# MEMBER C: Chart 3 — Correlation Heatmap
# Shows how stocks move together
# Red = move together (less diversification benefit)
# Green = move independently (great for diversification)

corr = stock_returns.corr()

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title('Stock Correlation Matrix', fontsize=16)
plt.tight_layout()
plt.savefig('chart_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart_correlation.png')

In [ ]:
# MEMBER C: Chart 4 — Technical Indicators Example (for one stock)

ex = indicators_dict['AAPL'].dropna()

fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

# Price + Bollinger Bands
axes[0].plot(ex.index, ex['Close'], label='Price', linewidth=1.5)
axes[0].plot(ex.index, ex['SMA_20'], label='SMA 20', alpha=0.7)
axes[0].fill_between(ex.index, ex['BB_Upper'], ex['BB_Lower'], alpha=0.1, color='gray', label='Bollinger Bands')
axes[0].set_title('AAPL — Price & Moving Averages', fontsize=14)
axes[0].legend(loc='upper left')
axes[0].set_ylabel('Price ($)')

# RSI
axes[1].plot(ex.index, ex['RSI'], color='purple')
axes[1].axhline(y=70, color='red', linestyle='--', label='Overbought (70)')
axes[1].axhline(y=30, color='green', linestyle='--', label='Oversold (30)')
axes[1].fill_between(ex.index, 70, 100, alpha=0.1, color='red')
axes[1].fill_between(ex.index, 0, 30, alpha=0.1, color='green')
axes[1].set_title('RSI', fontsize=14)
axes[1].set_ylim(0, 100)
axes[1].legend(loc='upper left')

# MACD
axes[2].plot(ex.index, ex['MACD'], label='MACD')
axes[2].plot(ex.index, ex['MACD_Signal'], label='Signal')
colors = ['green' if v >= 0 else 'red' for v in ex['MACD_Hist']]
axes[2].bar(ex.index, ex['MACD_Hist'], color=colors, alpha=0.3, width=1)
axes[2].set_title('MACD', fontsize=14)
axes[2].legend(loc='upper left')

plt.tight_layout()
plt.savefig('chart_indicators.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart_indicators.png')

## Task C2: Grand Comparison Dashboard

This is the **most important visualization** — it shows all strategies side by side.

We need:
1. A comparison **table** with all metrics
2. A **4-panel chart** showing portfolio value, Sharpe ratio, max drawdown, and annual return

In [ ]:
# MEMBER C: Grand Comparison Table
# Collect all metrics from Member B's results

all_metrics_list = [bh_metrics, mvo_metrics, minvol_metrics, equal_metrics]

# Add RL metrics if available
if 'rl_metrics_list' in dir() and rl_metrics_list:
    all_metrics_list.extend(rl_metrics_list)

# Add LLM metrics if available
if 'llm_metrics' in dir() and llm_metrics:
    all_metrics_list.append(llm_metrics)

all_metrics = pd.DataFrame(all_metrics_list).set_index('Strategy')

print('GRAND COMPARISON — ALL STRATEGIES')
print('=' * 90)
all_metrics

In [ ]:
# MEMBER C: 4-Panel Dashboard

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Panel 1: Portfolio Value Over Time
ax = axes[0, 0]
ax.plot(bh_portfolio.index, bh_portfolio.values, label='Buy & Hold SPY', linewidth=2, linestyle='--', color='black')
ax.plot(mvo_portfolio.index, mvo_portfolio.values, label='MVO (Max Sharpe)', linewidth=2, color='red')
ax.plot(equal_portfolio.index, equal_portfolio.values, label='Equal Weight', linewidth=1.5, linestyle='-.', color='gray')
if 'rl_results' in dir():
    for name, df_acc in rl_results.items():
        ax.plot(df_acc['date'], df_acc['account_value'], label=f'RL-{name.upper()}', linewidth=2)
if 'llm_portfolio' in dir():
    ax.plot(llm_portfolio.index, llm_portfolio.values, label='LLM', linewidth=2, color='purple')
ax.set_title('Portfolio Value', fontsize=14)
ax.set_ylabel('Value ($)')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# Panel 2: Sharpe Ratio
ax = axes[0, 1]
colors = ['green' if x > 0 else 'red' for x in all_metrics['Sharpe Ratio']]
all_metrics['Sharpe Ratio'].plot(kind='barh', ax=ax, color=colors)
ax.set_title('Sharpe Ratio (Higher = Better)', fontsize=14)
ax.axvline(x=0, color='black', linewidth=0.5)

# Panel 3: Max Drawdown
ax = axes[1, 0]
all_metrics['Max Drawdown (%)'].plot(kind='barh', ax=ax, color='salmon')
ax.set_title('Max Drawdown (Closer to 0 = Better)', fontsize=14)

# Panel 4: Annual Return
ax = axes[1, 1]
colors = ['green' if x > 0 else 'red' for x in all_metrics['Annual Return (%)']]
all_metrics['Annual Return (%)'].plot(kind='barh', ax=ax, color=colors)
ax.set_title('Annual Return %', fontsize=14)
ax.axvline(x=0, color='black', linewidth=0.5)

plt.suptitle('Strategy Comparison Dashboard', fontsize=18, y=1.02)
plt.tight_layout()
plt.savefig('chart_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart_dashboard.png')

## Task C3: Risk Analysis

Risk analysis answers: **"How dangerous is each strategy?"**

We look at three things:

### 1. Drawdown Chart
- Shows how far the portfolio fell from its peak at every point in time
- Deeper drawdowns = more painful for investors

### 2. Rolling Sharpe Ratio
- Shows whether the strategy's risk-adjusted performance is stable or erratic
- A steady Sharpe is better than one that swings wildly

### 3. Value at Risk (VaR)
- Answers: "What's the worst loss I can expect on 95% of days?"
- Example: VaR = -2% means you won't lose more than 2% on most days

In [ ]:
# MEMBER C: Drawdown Analysis

def drawdown_series(returns):
    """Calculate drawdown (how far below the peak) at each point."""
    cum = (1 + returns).cumprod()
    return (cum / cum.cummax() - 1) * 100

fig, ax = plt.subplots(figsize=(14, 6))

dd_bh = drawdown_series(benchmark_test_returns)
dd_mvo = drawdown_series(mvo_test_returns)

ax.fill_between(dd_bh.index, dd_bh.values, 0, alpha=0.3, label='Buy & Hold SPY')
ax.fill_between(dd_mvo.index, dd_mvo.values, 0, alpha=0.3, label='MVO')

if 'llm_returns' in dir():
    dd_llm = drawdown_series(llm_returns)
    ax.plot(dd_llm.index, dd_llm.values, label='LLM', linewidth=2, color='purple')

ax.set_title('Drawdown — How Far Did Each Strategy Fall From Its Peak?', fontsize=14)
ax.set_ylabel('Drawdown %')
ax.set_xlabel('Date')
ax.legend()
plt.tight_layout()
plt.savefig('chart_drawdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart_drawdown.png')

In [ ]:
# MEMBER C: Rolling Sharpe Ratio

def rolling_sharpe(returns, window=60):
    """Compute rolling Sharpe Ratio over a window of trading days."""
    rm = returns.rolling(window).mean() * 252
    rs = returns.rolling(window).std() * np.sqrt(252)
    return (rm - 0.04) / rs

fig, ax = plt.subplots(figsize=(14, 6))

rs_bh = rolling_sharpe(benchmark_test_returns)
rs_mvo = rolling_sharpe(mvo_test_returns)

ax.plot(rs_bh.index, rs_bh.values, label='Buy & Hold SPY', linewidth=2, linestyle='--', color='black')
ax.plot(rs_mvo.index, rs_mvo.values, label='MVO', linewidth=2, color='red')

if 'llm_returns' in dir():
    rs_llm = rolling_sharpe(llm_returns)
    ax.plot(rs_llm.index, rs_llm.values, label='LLM', linewidth=2, color='purple')

ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax.axhline(y=1, color='green', linestyle=':', alpha=0.5, label='Good (Sharpe=1)')
ax.set_title('Rolling 60-Day Sharpe Ratio', fontsize=14)
ax.set_ylabel('Sharpe Ratio')
ax.legend()
plt.tight_layout()
plt.savefig('chart_rolling_sharpe.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart_rolling_sharpe.png')

In [ ]:
# MEMBER C: Value at Risk (VaR) and Conditional VaR

strategies = {
    'Buy & Hold SPY': benchmark_test_returns,
    'MVO (Max Sharpe)': mvo_test_returns,
    'Equal Weight': equal_test_returns,
}
if 'llm_returns' in dir():
    strategies['LLM'] = llm_returns

confidence = 0.95

print(f'RISK METRICS ({confidence*100:.0f}% confidence)')
print('=' * 70)
print(f'{"Strategy":<22} {"VaR (Daily %)":<16} {"CVaR (Daily %)":<16} {"Worst Day %":<14}')
print('-' * 70)

for name, rets in strategies.items():
    var = np.percentile(rets, (1 - confidence) * 100)
    cvar = rets[rets <= var].mean()
    worst = rets.min()
    print(f'{name:<22} {var*100:<16.2f} {cvar*100:<16.2f} {worst*100:<14.2f}')

print(f'\nVaR: On {confidence*100:.0f}% of days, losses won\'t exceed this amount.')
print('CVaR: When losses DO exceed VaR (worst 5% of days), this is the average loss.')

## Task C4: Efficient Frontier Plot

The **Efficient Frontier** shows all possible portfolio combinations and highlights the best ones.

- Each dot = one possible way to split money across stocks
- The upper-left edge = the "frontier" (best possible trade-off between risk and return)
- Stars = our optimal portfolios

In [ ]:
# MEMBER C: Efficient Frontier

n_portfolios = 10000
np.random.seed(42)
W = np.random.dirichlet(np.ones(n_assets), size=n_portfolios)

# Vectorized computation (fast!)
port_rets = W @ expected_returns.values * 100
port_vols = np.sqrt(np.einsum('ij,jk,ik->i', W, cov_matrix.values, W)) * 100
port_sharpes = (port_rets - 4) / port_vols

fig, ax = plt.subplots(figsize=(14, 8))
scatter = ax.scatter(port_vols, port_rets, c=port_sharpes, cmap='RdYlGn', s=5, alpha=0.5)
plt.colorbar(scatter, label='Sharpe Ratio')

# Mark optimal portfolios
ax.scatter(portfolio_volatility(mvo_weights)*100, portfolio_return(mvo_weights)*100,
           color='red', marker='*', s=300, zorder=5, label='Max Sharpe')
ax.scatter(portfolio_volatility(minvol_weights)*100, portfolio_return(minvol_weights)*100,
           color='blue', marker='*', s=300, zorder=5, label='Min Volatility')
ax.scatter(portfolio_volatility(equal_weights)*100, portfolio_return(equal_weights)*100,
           color='orange', marker='D', s=150, zorder=5, label='Equal Weight')

ax.set_title('Efficient Frontier — 10,000 Random Portfolios', fontsize=16)
ax.set_xlabel('Annual Volatility (Risk) %')
ax.set_ylabel('Annual Return %')
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig('chart_efficient_frontier.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart_efficient_frontier.png')

## Task C5: Write the Report

### Report Structure (max 20 pages)

Use the charts you saved (`.png` files) in your report.

| Section | Pages | What to Write | Charts to Include |
|---------|-------|---------------|-------------------|
| **Executive Summary** | 1 | Brief overview: what we did, key results | None |
| **Introduction & Motivation** | 1-2 | Why algorithmic trading? Why our chosen approach? | None |
| **Strategy Design** | 2-3 | How each strategy works (MVO, RL/LLM) | None (diagrams optional) |
| **Implementation Details** | 2-3 | Tools used, data sources, time period justification | None |
| **EDA & Data Analysis** | 2-3 | Describe the data, key observations | `chart_normalized_prices.png`, `chart_risk_return.png`, `chart_correlation.png`, `chart_indicators.png` |
| **Backtesting Results** | 3-4 | Performance comparison, what worked, what didn't | `chart_dashboard.png`, `chart_efficient_frontier.png`, comparison table |
| **Risk Analysis** | 2 | Drawdown analysis, VaR, rolling Sharpe | `chart_drawdown.png`, `chart_rolling_sharpe.png`, VaR table |
| **Paper Trading** | 1 | Alpaca setup, initial results (screenshots) | Screenshots from Alpaca |
| **Discussion** | 1-2 | Strengths, weaknesses, limitations, improvements | None |
| **Conclusion** | 1 | Key takeaways | None |

### Writing Tips
- Use **past tense** for what you did ("We implemented...")
- **Explain WHY**, not just what. "We chose 15 stocks because..."
- Be **honest about limitations**. The graders reward critical thinking!
- **Don't fake results**. Only 10% of the grade is performance.
- Include the **comparison table** from the Grand Comparison cell above

## Task C6: Follow-Up Paper Trading Report (due April 7)

### Structure (max 5 pages)

| Section | Pages | Content |
|---------|-------|---------|
| Overview | 0.5 | Which strategy we paper traded, when we started |
| Results | 2 | Daily P&L, portfolio value chart, comparison to benchmark |
| Analysis | 1.5 | Did it behave as expected? Any surprises? |
| Conclusion | 1 | Lessons learned from live(ish) trading |

### How to Get Paper Trading Data
Run the cell below periodically to collect data from Alpaca.

In [ ]:
# MEMBER C: Collect paper trading data for the follow-up report
# Run this cell once a day to log your portfolio value

if alpaca is not None:
    account = alpaca.get_account()
    positions = alpaca.list_positions()
    
    print(f'Date: {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")}')
    print(f'Portfolio Value: ${float(account.portfolio_value):,.2f}')
    print(f'Cash: ${float(account.cash):,.2f}')
    print(f'P&L Today: ${float(account.equity) - float(account.last_equity):,.2f}')
    
    if positions:
        print(f'\nPositions ({len(positions)}):')
        for p in positions:
            print(f'  {p.symbol}: {p.qty} shares, P&L: ${float(p.unrealized_pl):,.2f} ({float(p.unrealized_plpc)*100:.1f}%)')
    
    # Save to CSV for tracking over time
    log_entry = pd.DataFrame([{
        'date': pd.Timestamp.now(),
        'portfolio_value': float(account.portfolio_value),
        'cash': float(account.cash),
    }])
    
    log_file = 'paper_trading_log.csv'
    try:
        existing = pd.read_csv(log_file)
        log_entry = pd.concat([existing, log_entry], ignore_index=True)
    except FileNotFoundError:
        pass
    log_entry.to_csv(log_file, index=False)
    print(f'\nLogged to {log_file}')
else:
    print('Alpaca not connected. Set up API keys to track paper trading.')

---
---
# SHARED: Final Checklist Before Submission
---
---

## Member A Checklist
- [ ] All data downloads work without errors
- [ ] No API keys are visible in the notebook
- [ ] Notebook runs top-to-bottom (Kernel → Restart & Run All)
- [ ] Alpaca paper trading is running

## Member B Checklist
- [ ] At least 2 benchmarks implemented (Buy & Hold + MVO)
- [ ] At least 1 main strategy implemented (RL or LLM)
- [ ] All strategies backtested on test data
- [ ] Results compared against S&P 500

## Member C Checklist
- [ ] All charts saved as PNG files
- [ ] Grand comparison table is complete
- [ ] Risk analysis (drawdown, VaR) is done
- [ ] Report is written (max 20 pages, PDF)
- [ ] Report includes: Executive Summary, Introduction, Strategy Design, Implementation, Backtest Results, Discussion, Conclusion

## Deadlines
- **March 31, 2026**: Main report + notebook
- **April 7, 2026**: Paper trading follow-up report